# 텍스트 임베딩

이 노트북에서는 **건강/피트니스** 주제를 예시로 텍스트 임베딩을 다룹니다.

1. `EmbeddingsClient`를 초기화합니다.
2. 건강 테마 문구를 임베딩합니다.
3. 템플릿 기반 전처리 임베딩을 확인합니다.

> 안내: 이 노트북은 교육 목적의 예제입니다. 의학적 조언이 필요하면 전문가와 상담하세요.

## 1. 설정 및 환경 구성

### 사전 준비 사항
- Microsoft Foundry에 텍스트 임베딩 모델(`text-embedding-3-small`)을 배포하세요.

라이브러리를 임포트하고 다음 환경 변수를 불러옵니다.
- `ENDPOINT`: 프로젝트 엔드포인트
- `API_KEY`: API 키
- `TEXT_EMBEDDING_MODEL`: 텍스트 임베딩 모델 배포 이름

이후 `EmbeddingsClient`를 생성합니다.

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

from azure.core.credentials import AzureKeyCredential
from azure.ai.inference import EmbeddingsClient

# 환경 변수 로드
def _clean_env(value: str | None) -> str | None:
    if value is None:
        return None
    return value.strip().strip("\"").strip("'")

dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd().parent / '.env',
    Path('.env'),
    Path('..') / '.env',
]
dotenv_path = next((p for p in dotenv_candidates if p.exists()), None)
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print(f"✅ .env 로드 완료: {dotenv_path}")
else:
    print("⚠️ .env 파일을 찾지 못했습니다. 현재/상위 폴더를 확인하세요.")

# 환경 변수 읽기
endpoint = _clean_env(os.environ.get("ENDPOINT"))
api_key = _clean_env(os.environ.get("API_KEY"))
embedding_model = _clean_env(os.environ.get("TEXT_EMBEDDING_MODEL"))

# 임베딩 클라이언트 초기화
embeddings_client = None
missing_keys = [
    name
    for name, value in {
        "ENDPOINT": endpoint,
        "API_KEY": api_key,
        "TEXT_EMBEDDING_MODEL": embedding_model,
    }.items()
    if not value
]
if missing_keys:
    print(f"❌ 필수 환경 변수가 없습니다: {', '.join(missing_keys)}")
else:
    try:
        embeddings_client = EmbeddingsClient(
            endpoint=endpoint,
            credential=AzureKeyCredential(api_key),
            model=embedding_model
        )
        print("✅ EmbeddingsClient 생성 완료")
    except Exception as e:
        print("❌ EmbeddingsClient 생성 오류:", e)

## 2. 텍스트 임베딩

`EmbeddingsClient`로 문장을 벡터로 변환해 의미 기반 비교가 가능하도록 만듭니다.  

아래 3개 문구는 바로 다음 코드 셀의 `text_phrases`와 동일합니다:

- "하루 20분 걷기는 심폐 지구력 향상에 도움이 됩니다"
- "집에서 하는 15분 전신 HIIT 루틴"
- "스트레스 완화를 위한 마음챙김 호흡 운동"

출력 결과는 각 문장을 의미론적 공간(semantic space)에서 표현하는 숫자 벡터입니다.  
이제 임베딩 결과를 확인해봅시다.


In [ ]:
text_phrases = [
    "하루 20분 걷기는 심폐 지구력 향상에 도움이 됩니다",
    "집에서 하는 15분 전신 HIIT 루틴",
    "스트레스 완화를 위한 마음챙김 호흡 운동"
]

try:
    response = embeddings_client.embed(
        input=text_phrases
    )
    for item in response.data:
        length = len(item.embedding)
        print(
            f"data[{item.index}]: length={length}, "
            f"[{item.embedding[0]}, {item.embedding[1]}, "
            f"..., {item.embedding[length-2]}, {item.embedding[length-1]}]"
        )

    print(response.usage)

except Exception as e:
    print("❌ 텍스트 임베딩 중 오류:", e)

## 3. 프롬프트 템플릿 예제

이번 노트북의 초점은 임베딩이지만, 사용자 텍스트 앞에 컨텍스트를 덧붙이는 방식도 함께 소개합니다.

> 당신은 HealthFitKorea, 한국 사용자를 위한 피트니스 가이드 모델입니다.
실천 가능한 건강 조언에 집중하고, 의료 전문가는 아니라는 점을 안내하세요.
사용자 메시지:

이런 전처리는 도메인 맥락을 반영한 임베딩을 만들 때 도움이 될 수 있습니다.

In [ ]:
# 시스템 템플릿을 사용자 텍스트 앞에 붙여 임베딩합니다.
TEMPLATE_SYSTEM = (
    "당신은 HealthFitKorea, 한국 사용자를 위한 피트니스 가이드 모델입니다.\n"
    "실천 가능한 건강 조언에 집중하고, 의료 전문가는 아니라는 점을 안내하세요.\n\n"
    "사용자 메시지:"
)

def embed_with_template(user_text):
    """시스템 템플릿을 앞에 붙여 사용자 텍스트를 임베딩합니다."""
    if embeddings_client is None:
        raise RuntimeError("embeddings_client is not initialized. Check ENDPOINT/API_KEY/TEXT_EMBEDDING_MODEL in .env")

    content = TEMPLATE_SYSTEM + " " + user_text
    response = embeddings_client.embed(
        input=[content]
    )
    return response.data[0].embedding

sample_user_text = "시간이 부족한 직장인을 위한 10분 홈트 루틴을 추천해 주세요."
embedding_result = embed_with_template(sample_user_text)

print("임베딩 길이:", len(embedding_result))
print("앞 8개 차원:", embedding_result[:8])